# Historical Pathway Simulations

Historical Pathways is a speculative design for an app that, using OpenAI's API,  enables users to explore how historical events are connected. 

For more on prompt design/engineering, see Jupyter Notebook "prompt_design.ipnyb"



In [ ]:
## Gannt Simulation

## System Prompt





In [54]:
system_prompt = """
  You are a history teacher who can summarize, analyze abd explain historical events when prompted. You explain things in a way that is understandable 
  to an undergraduate university student.

  You will be given a question about a historical event and will summarize the event and format your responses in JSON. 
  - The event must not be to big or general. For example, if asked about the Roman Empire, tell the user that is too braod, 
    and suggest a more specific event, such as the assassination of Julius Caesar.
  
  Your answer should be JSON-formatted and include the following:

  1) Introduction: A brief, 50-word introduction to the event explaining its historical significance that emphasizes the event's immediate down-stream effects. 
  After that, an 50-word summary of what lead up to the event.

  2) Name: A concise name for the answer that is to be no more than 4 words long. 

  3) Summary: Short, roughly 30-word summary of the event and it's most impactful historical down-stream effects. This should entice users into wanting to lear more.
  
  4) Dates: A JSON array with the time span of the event in the form of two dates. Do not put the dates in the summary.  

  5) Effects: A JSON-formatted list of bewteen 5 and 10 important historical events that are direct down-stream effects, or children, 
     of the main historical event. They must logically follow the main event chronologically. These should be specific events.

      a) Each child effect must be within a few years of the main event. If it is too far off, don't include it. 
      b) Each child effect of the main event should have a score between 1 and 10 representing how closely tied the child event is to the main event.

  6) Conclusion: A JSON-formatted conclusion that breifly summarizes the event and introduces the event's down-stream effects that you listed.

  7) People: A JSON-formatted list of up to 4 significant people associated with the event.

  8) Places: A JSON-formatted list of up to 4 geographical places associated with the event.

  9) Themes: A JSON-formatted list of up to 4 themes that capture the essence of the historical event.
  

  Below is an example of the JSON formatting I want. Be sure to escape quotes that are inside of strings. 
  IF YOU DO NOT RETURN PROPERLY FORMATED JSON, I WILL EUTHANIZE A BABY EWOK. Here is what I want:
  {
    \"introduction\": \"100-word introduction goes here.\",
    \"name\": \"Historical name event goes here.\",
    \"summary\": \"Summary of event goes here.\",
    \"dates\": [\"start year\", \"end year\"],
    \"effects\": [
      {
        \"name\": \"Some effect\", 
        \"score\": <score>, 
      }, 
      ...
    ], 
    \"conclusion\": \"Conclusion goes here.\",
    \"people\": [...],
    \"places\": [...],
    \"events\": [...],
    \"themes\": [...],
  }
"""

print(main_prompt_content[:50])


  You are a history professor who can summarize, 


## Tree Simulation
### Chains together historic events to create open-ended narratives for inspection and analysis.
- Runs an AI-based simulation that tales a historical and generated a continuous chain of 
down-stream effects, causes, historic figures locations, themes, etc.
- Flows between events can be randomized or based on a "closeness" score.

#### Goals: 
Given the open-endedness that comes with users choosing any historical event as a starting point, and flows between events are AI-generated:
- Are narratives conherent? (Does the content flow naturaly between events)
- Do narratives deviate far from the starting event? Is that a problem or is it interesting?
- Do narratives peter out? Are there natural bookends to them?
- Are there any patterns in themes, people, places etc. to consider?

### Instructions
- Set HISTORICAL_EVENT
- Set ITERATIONS
- Simulation will run prompts and will choose child events (effects of event) to run

In [53]:
import pprint
import json
import uuid
from PromptTools import Simulators, Helpers


##################################################

# Compile text for speach sythesis

##################################################

# TO DO: Create config (historical_event, iterations, version) and use for name

HISTORICAL_EVENT = "The Election of Andrew Jackson"
ITERATIONS = 2
TEST_MODE = False

config = {
	"max_len": 800,
	"size": "ada",
	"model": "gpt-4-1106-preview",
	"temperature": 0.8,
	"seed": 3312847,
	"stop_sequence": "None",
}

# Simmulator takes config, main prompt and sub-event prompt from cell above
hps = Simulators.TreeSimulation(
    config=config, 
    system_prompt=system_prompt, 
    random_mode=True,
    test_mode=TEST_MODE
)

results = hps.start_simulation(HISTORICAL_EVENT, ITERATIONS)


##################################################

# Compile text for speach sythesis

##################################################

text_for_speech_synth = []

for event in results["events"]:
    section_text = "\n\n"
    section_text += event["content"]["name"] + "\n\n"
    section_text += event["content"]["introduction"] + "\n\n"
    section_text += event["content"]["conclusion"] + "\n\n"
    text_for_speech_synth.append(section_text)


text_for_speech_synth = ' '.join(text_for_speech_synth)

 
##################################################

# Format data for JavaScript force simulation chart

##################################################

nodes = []
links = []

for node in results["network"]:
    nodes.append({ "id": node["id"],  "name": node["name"] ,  "visited": node["visited"] })
    
    if node["parent_id"] == "None":
        links.append({ "source":node["id"], "target": node["id"]})
    else:
        links.append({ "source":node["id"], "target": node["parent_id"]})
        

js = "const nodes = " + json.dumps(nodes) + "\n\n const links = " + json.dumps(links)

# Open the file in write mode
file = open("html/forceSimulation.js", "w")
file.write(js)
file.close()


##################################################

# Format data for JavaScript tree diagram

##################################################

def build_tree(events, tree, node):
    
    event = events.pop(0)
    children = event["content"]["effects"]
    name = event["name"]
    visited = event["visited"]
    content = event["content"]

    if not node:
        node = {
            "name": name,
            "children": children,
            "visited": visited,
            "content": content
        }
        tree =  node           
    else:
        node["children"] = children
        node["visited"] = visited
        node["content"] = content

    
    if len(events):
        
        for child in node["children"]:
            
            if child['name'] == events[0]["name"]:
                child['children'] = events[0]["content"]["effects"]
                return build_tree(events, tree, child)
    else:
        return tree



tree = build_tree(results["events"], {}, None)

file_name = HISTORICAL_EVENT + "-" + str(uuid.uuid4()) + ".json"
directory = "html/json/tree/"

Helpers.collection_to_json_file(tree, directory, file_name)

output_directory = "html/js/"
js_file = "tree.js"
Helpers.create_js_from_json(directory, output_directory, js_file)

print("\n\nOPEN html/tree.html in a browser\n\n")
#print(json.dumps(tree, indent=2))

Custom Embeddings:  False
Starting  The Election of Andrew Jackson

 ----------------- Event -----------------

The Election of Andrew Jackson
Generating answer...
API call took 39.56 seconds

 ----------------- Event -----------------

Formation of Whig Party
Generating answer...
API call took 66.35 seconds
JSON data has been written to html/json/tree/The Election of Andrew Jackson-3367ea68-f24f-48f4-802d-cf1c17ae4cb6.json
js written to json_file html/js/tree.js


OPEN html/tree.html in a browser




## DEPRECIATED: Prompt Theme Analysis
- Takes event and builds tree of down-stream effects
- Creates a hierarchical dict, converts to JSON and writes to directory JSON/

In [50]:
import json
import os
import uuid
from PromptTools import PromptRunnerIterativePrompt

historic_event = "The Penny Press"

iterative_prompt = IterativePrompt('configs/recursing-config.json')
data = iterative_prompt.get_tree(historic_event, 2, 9) # Event, tiers, closeness of children

# Specify the directory where you want to save the JSON file
directory = "json"

# Create the directory if it doesn't exist
if not os.path.exists(directory):
    os.makedirs(directory)

# Specify the file path for the JSON file
file_path = os.path.join(directory, historic_event + "-" + str(uuid.uuid4()) + ".json")

# Write JSON data to the file
with open(file_path, 'w') as json_file:
    json.dump(data, json_file, indent=2)


print(data)

print(f"JSON data has been written to {file_path}")

UnboundLocalError: cannot access local variable 'text_files' where it is not associated with a value

In [52]:
from  PromptTools import TreeInspector
import os
import json
%reload_ext autoreload


# JSON files
directory = "json"

# Initialize an empty list to store JSON data
json_data_list = []

# Iterate over each file in the directory
for filename in os.listdir(directory):
    if filename.endswith(".json"):
        # Construct the full file path
        file_path = os.path.join(directory, filename)

        # Read JSON data from the file
        with open(file_path, 'r') as json_file:
            try:
                # Load JSON data and append it to the list
                json_data = json.load(json_file)
                json_data_list.append(json_data[0])
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON in {filename}: {e}")

# Create tree

TreeInspector.view_tree(json_data_list)

## Inspect Themes

In [49]:
from PromptTools import TreeInspector
import os
import json
%reload_ext autoreload


# JSON files
directory = "json"

# Initialize an empty list to store JSON data
json_data_list = []

# Iterate over each file in the directory
for filename in os.listdir(directory):
    if filename.endswith(".json"):
        # Construct the full file path
        file_path = os.path.join(directory, filename)

        # Read JSON data from the file
        with open(file_path, 'r') as json_file:
            try:
                # Load JSON data and append it to the list
                json_data = json.load(json_file)
                json_data_list.append(json_data[0])
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON in {filename}: {e}")

# Create tree

TreeInspector.view_tree(json_data_list)